In [4]:
import ROOT
import numpy as np
import pandas as pd

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    dx=10e-6
    return (model(p + dx, par) - model(p, par)) / dx

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati (2)")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy()

#definizione variabili
p = df[:, 6]
q = df[:, 7]
err_p = df[:, 8]
err_q = df[:,9]

#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParLimits(0, 0, 50) 
model_fit.SetParLimits(1, 0, 50)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati con errori; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerColor(ROOT.kBlue)
gr1.SetLineColor(ROOT.kBlack)

fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("APX")

c1.Draw()
c1.SaveAs("fit_root.png")

Fit status: 0****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  1.37882e+07
NDf                       =          121
Edm                       =  7.00125e-06
NCalls                    =          137
A                         =  1.41553e-12   +/-   2.01043e-06  	 (limited)
B                         =      11.4388   +/-   0.0205206    	 (limited)
C                         =        94.22   +/-   0.00409754  

Parametri stimati:
A = 1.4155343563970746e-12 ± 2.0104318668168197e-06
B = 11.438783375256195 ± 0.020520609378359644
C = 94.22003757994224 ± 0.004097535295827275
Chi2 = 13788221.917740377


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


In [5]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if par[0] < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if par[1] < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B NON è compatibile con 0


In [6]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff

print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kGreen+2)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kBlue)
hist.SetFillColor(ROOT.kBlue)
hist.GetXaxis().SetTitle("valore")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "SR")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

c2.Draw()
c2.SaveAs("hist.png")

[ 9.54044836e+02  9.62540079e+02  9.38232485e+02  9.38390550e+02
  9.09223382e+02  9.03784927e+02  8.86249554e+02  8.76922602e+02
  8.86568411e+02  8.83590515e+02  8.89397895e+02  9.06360277e+02
  9.24565770e+02  9.51590309e+02  9.86539781e+02  1.03269510e+03
  1.09763220e+03  1.18514136e+03  1.31418210e+03  1.49632918e+03
  1.80595647e+03  2.36516723e+03  3.73596509e+03  1.16728087e+04
 -7.79438924e+03 -2.62673593e+03 -1.47431435e+03 -9.69007396e+02
 -6.88125593e+02 -5.10351919e+02 -3.91637197e+02 -2.99249703e+02
 -2.32491705e+02 -1.81219945e+02 -1.40050817e+02 -1.08295762e+02
 -8.21589458e+01 -6.26845209e+01 -4.06082732e+01 -2.58784013e+01
 -1.44673104e+01 -1.93989466e+00  5.36521555e+00  1.51931581e+01
  2.32205334e+01  2.72201293e+01  3.83776195e+01  4.20816832e+01
  4.99145084e+01  5.69979468e+01  5.67345079e+01  5.90003571e+01
  5.94598705e+01  6.17449479e+01  6.14441187e+01  5.81267395e+01
  5.72178642e+01  5.98552315e+01  5.78187896e+01  5.94350652e+01
  5.78022853e+01  5.85630

Info in <TCanvas::Print>: png file hist.png has been created


In [7]:
if parameters[0] - 1 < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if parameters[1] < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if parameters[2] - 1 < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const è compatibile con 1
---
mu è compatibile con 0
---
sigma è compatibile con 1


In [8]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Dati con errori; x; y")
gr2.SetMarkerStyle(20)
gr2.SetMarkerColor(ROOT.kBlue)
gr2.SetLineColor(ROOT.kBlack)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")


c3.Draw()
c3.SaveAs("fit_line.png")

Fit status: 0
Parametri stimati:
m = -0.9582207655569372 ± 0.0006059688551759639
q = 634.2274069263655 ± 0.23136647433940463
Chi2 = 243717059.16256598
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =  2.43717e+08
NDf                       =          122
Edm                       =  1.01852e-16
NCalls                    =           31
m                         =    -0.958221   +/-   0.000605969 
q                         =      634.227   +/-   0.231366    


Info in <TCanvas::Print>: png file fit_line.png has been created
